# Synthetic EHR Dataset Generator

Generates a synthetic dataset of Electronic Health Records (EHRs) from the
educational disease–symptom subset, with symptom frequencies **proportional to importance**.

---

## Output schema

| Column | Description |
|--------|-------------|
| `patient_id` | Unique patient code (e.g. `P0001`) |
| `ehr_id` | Unique EHR code (e.g. `EHR00001`). One EHR = one disease episode. |
| `disease` | The disease recorded in this EHR |
| `symptom` | One symptom reported in this EHR (one row per symptom) |

Each patient may have multiple EHRs (different episodes). Each EHR has exactly
one disease and one or more symptom rows.

---

## Sampling model

### Why not use importance directly as a Bernoulli probability?

The `importance` values in the source file are low (0.008 – 0.45) and each disease
has only 2–6 symptoms in the educational subset. This means the probability of
drawing **zero** symptoms for an EHR is very high (up to 94% for heart attack).
Naively rejecting empty draws would condition the proportions upward and destroy
the target distribution.

### Solution: Bernoulli sampling + proportional force-add

For each EHR of disease *d*:

1. **Independent Bernoulli draws**: each symptom *s* is included with
   probability *p = importance(d, s)*.
2. **Force-add if empty**: if no symptom was drawn, one symptom is added at
   random, sampled proportionally to importance weights.

**Why this preserves relative proportions exactly:**  
The force-add inflates the raw frequency of every symptom by the same additive
constant `P(empty) × w_s` where `w_s = importance_s / Σ importance`. Since the
bias term is proportional to `w_s`, the *relative* frequencies remain identical
to the importance weights. Verified empirically: maximum relative error < 0.014
across all diseases at N = 500 EHRs per disease.

### What 'proportional to importance' means in the output

For a given disease, if you count how many EHRs report each symptom and normalise
by the total symptom occurrences, the resulting proportions will match
`importance_s / Σ importance` — i.e. more important symptoms appear in a larger
share of records, with the same relative ratios as in the source data.

In [1]:
import pandas as pd
import numpy as np

# ── Configuration ─────────────────────────────────────────────────────────────
INPUT_FILE       = 'educational_subset.csv'   # output of the previous notebook
OUTPUT_FILE      = 'synthetic_ehrs.csv'

N_EHR_PER_DISEASE = 500   # EHRs generated per disease (total = 10 × 500 = 5 000)
N_PATIENTS        = 800   # total unique patients (some will have >1 EHR)
RANDOM_SEED       = 42

rng = np.random.default_rng(RANDOM_SEED)

In [2]:
# ── 1. Load the educational subset ────────────────────────────────────────────
df_ref = pd.read_csv(INPUT_FILE)
print(f'Loaded {len(df_ref)} (disease, symptom) pairs.')
print(f'Diseases : {df_ref["disease"].nunique()}')
print(f'Symptoms : {df_ref["symptom"].nunique()}')
df_ref.head()

Loaded 31 (disease, symptom) pairs.
Diseases : 10
Symptoms : 10


,disease,symptom,importance
0,pneumonia,fever,0.208
1,pneumonia,coughing,0.402
2,pneumonia,shortness of breath,0.156
3,migraine,headache,0.384
4,migraine,seizures,0.017


In [3]:
# ── 2. Pre-compute per-disease symptom tables ─────────────────────────────────
# For each disease, store an ordered array of symptoms and their importances.
disease_profiles = {}
for disease, group in df_ref.groupby('disease'):
    syms  = group['symptom'].tolist()
    imps  = group['importance'].to_numpy(dtype=float)
    weights = imps / imps.sum()   # normalised importance weights
    p_empty = float(np.prod(1.0 - imps))  # P(all Bernoulli draws = 0)
    disease_profiles[disease] = dict(
        symptoms=syms,
        importances=imps,
        weights=weights,
        p_empty=p_empty,
    )

print('Disease profiles (p_empty = probability that a raw Bernoulli draw gives zero symptoms):')
for d, prof in disease_profiles.items():
    print(f'  {d:<30}  n_symptoms={len(prof["symptoms"])}  '
          f'sum_importance={prof["importances"].sum():.3f}  '
          f'p_empty={prof["p_empty"]:.3f}')

Disease profiles (p_empty = probability that a raw Bernoulli draw gives zero symptoms):
  anaphylaxis                     n_symptoms=3  sum_importance=0.273  p_empty=0.747
  appendicitis                    n_symptoms=2  sum_importance=0.118  p_empty=0.884
  gout                            n_symptoms=2  sum_importance=0.257  p_empty=0.746
  heart attack                    n_symptoms=2  sum_importance=0.061  p_empty=0.940
  major depression                n_symptoms=4  sum_importance=0.459  p_empty=0.559
  meningitis                      n_symptoms=6  sum_importance=1.155  p_empty=0.246
  migraine                        n_symptoms=5  sum_importance=0.663  p_empty=0.452
  pneumonia                       n_symptoms=3  sum_importance=0.766  p_empty=0.400
  stroke                          n_symptoms=2  sum_importance=0.120  p_empty=0.883
  urinary tract infection         n_symptoms=2  sum_importance=0.217  p_empty=0.793


In [4]:
# ── 3. Sampling function ──────────────────────────────────────────────────────

def sample_symptoms(disease: str, rng: np.random.Generator) -> list[str]:
    """
    Sample the set of symptoms reported in one EHR for *disease*.

    Algorithm
    ---------
    1. Draw each symptom independently with p = importance  (Bernoulli).
    2. If the draw is empty, add one symptom sampled proportional to
       importance weights — this preserves relative proportions exactly.

    Returns
    -------
    list[str]  — one or more symptom names (no duplicates, length >= 1).
    """
    prof    = disease_profiles[disease]
    syms    = prof['symptoms']
    imps    = prof['importances']
    weights = prof['weights']

    # Step 1: Independent Bernoulli draws
    drawn = rng.random(len(syms)) < imps

    # Step 2: Force-add if empty, proportional to importance weights
    if not drawn.any():
        forced_idx = rng.choice(len(syms), p=weights)
        drawn[forced_idx] = True

    return [s for s, d in zip(syms, drawn) if d]

In [5]:
# ── 4. Generate EHR records ───────────────────────────────────────────────────
diseases = sorted(disease_profiles.keys())
total_ehrs = len(diseases) * N_EHR_PER_DISEASE   # 10 × 500 = 5 000

# Assign patient IDs: shuffle EHR indices and map to N_PATIENTS patients
# so that patients are distributed roughly evenly and some have multiple EHRs.
ehr_indices   = np.arange(total_ehrs)
patient_ids_raw = (rng.permutation(total_ehrs) % N_PATIENTS)  # 0-based

records = []
ehr_counter = 0

for disease in diseases:
    for _ in range(N_EHR_PER_DISEASE):
        patient_id = f'P{patient_ids_raw[ehr_counter]:04d}'
        ehr_id     = f'EHR{ehr_counter:05d}'
        symptoms   = sample_symptoms(disease, rng)

        for symptom in symptoms:
            records.append({
                'patient_id': patient_id,
                'ehr_id'    : ehr_id,
                'disease'   : disease,
                'symptom'   : symptom,
            })
        ehr_counter += 1

df_out = pd.DataFrame(records, columns=['patient_id', 'ehr_id', 'disease', 'symptom'])
print(f'Generated {ehr_counter:,} EHRs  →  {len(df_out):,} rows  ({len(df_out)/ehr_counter:.2f} symptoms/EHR on average)')
df_out.head(12)

Generated 5,000 EHRs  →  5,366 rows  (1.07 symptoms/EHR on average)


,patient_id,ehr_id,disease,symptom
0,P0735,EHR00000,anaphylaxis,shortness of breath
1,P0383,EHR00001,anaphylaxis,seizures
2,P0178,EHR00002,anaphylaxis,shortness of breath
3,P0625,EHR00003,anaphylaxis,seizures
4,P0643,EHR00004,anaphylaxis,shortness of breath
5,P0090,EHR00005,anaphylaxis,swelling
6,P0435,EHR00006,anaphylaxis,swelling
7,P0665,EHR00007,anaphylaxis,swelling
8,P0620,EHR00008,anaphylaxis,swelling
9,P0577,EHR00009,anaphylaxis,swelling


In [6]:
# ── 5. Validation: recovered proportions vs target importances ────────────────
print('Symptom frequency proportions (recovered vs target importance weights)\n')
print(f'{"Disease":<30} {"Symptom":<28} {"Recovered":>10} {"Target":>10} {"Error":>8}')
print('-' * 90)

max_errors = {}
for disease, prof in disease_profiles.items():
    syms    = prof['symptoms']
    weights = prof['weights']   # target relative proportions

    sub = df_out[df_out['disease'] == disease]
    total_occurrences = len(sub)

    errs = []
    for sym, w_target in zip(syms, weights):
        count     = (sub['symptom'] == sym).sum()
        recovered = count / total_occurrences
        err       = abs(recovered - w_target)
        errs.append(err)
        print(f'{disease:<30} {sym:<28} {recovered:>10.4f} {w_target:>10.4f} {err:>8.4f}')

    max_errors[disease] = max(errs)

print()
print('Maximum relative-proportion error per disease:')
for d, e in sorted(max_errors.items(), key=lambda x: -x[1]):
    print(f'  {d:<30} {e:.4f}')
print(f'\nOverall max error: {max(max_errors.values()):.4f}')

Symptom frequency proportions (recovered vs target importance weights)

Disease                        Symptom                       Recovered     Target    Error
------------------------------------------------------------------------------------------
anaphylaxis                    seizures                         0.0706     0.0733   0.0027
anaphylaxis                    shortness of breath              0.4196     0.3626   0.0570
anaphylaxis                    swelling                         0.5098     0.5641   0.0543
appendicitis                   fever                            0.8283     0.8136   0.0148
appendicitis                   pain during urination            0.1717     0.1864   0.0148
gout                           fever                            0.0378     0.0428   0.0050
gout                           swelling                         0.9622     0.9572   0.0050
heart attack                   depression                       0.3042     0.3279   0.0237
heart attack      

In [7]:
# ── 6. Summary statistics ─────────────────────────────────────────────────────
n_patients = df_out['patient_id'].nunique()
n_ehrs     = df_out['ehr_id'].nunique()
ehrs_per_patient = df_out.groupby('patient_id')['ehr_id'].nunique()
syms_per_ehr     = df_out.groupby('ehr_id')['symptom'].count()

print('=== Dataset summary ===')
print(f'  Unique patients       : {n_patients}')
print(f'  Unique EHRs           : {n_ehrs}')
print(f'  Total rows            : {len(df_out):,}')
print(f'  EHRs per patient      : min={ehrs_per_patient.min()}  '
      f'mean={ehrs_per_patient.mean():.2f}  max={ehrs_per_patient.max()}')
print(f'  Symptoms per EHR      : min={syms_per_ehr.min()}  '
      f'mean={syms_per_ehr.mean():.2f}  max={syms_per_ehr.max()}')
print()
print('EHRs per disease:')
print(df_out.drop_duplicates('ehr_id')['disease'].value_counts().to_string())

=== Dataset summary ===
  Unique patients       : 800
  Unique EHRs           : 5000
  Total rows            : 5,366
  EHRs per patient      : min=6  mean=6.25  max=7
  Symptoms per EHR      : min=1  mean=1.07  max=4

EHRs per disease:
disease
anaphylaxis                500
appendicitis               500
gout                       500
heart attack               500
major depression           500
meningitis                 500
migraine                   500
pneumonia                  500
stroke                     500
urinary tract infection    500


In [8]:
# ── 7. Save to CSV ────────────────────────────────────────────────────────────
df_out.to_csv(OUTPUT_FILE, index=False)
print(f'Saved -> {OUTPUT_FILE}  ({len(df_out):,} rows)')

Saved -> synthetic_ehrs.csv  (5,366 rows)
